# Deploy Rosetta SDL MCP Server to AgentCore

This notebook deploys the **Rosetta SDL MCP Server** to **Amazon Bedrock AgentCore**,
enabling any AI agent to discover data assets, query governed metrics, and search
documents through the semantic layer — all via the MCP protocol.

## Architecture

```
┌─────────────────────┐
│   Any AI Agent      │
│  (QuickSuite,       │
│   Strands, Claude)  │
└──────────┬──────────┘
           │ JWT Auth (Cognito)
           ▼
┌─────────────────────┐
│  AgentCore Gateway  │
│   (MCP Protocol)    │
└──────────┬──────────┘
           │ OAuth2 (Cognito M2M)
           ▼
┌─────────────────────┐
│  AgentCore Runtime  │
│  Rosetta SDL MCP    │
│  (8 tools)          │
└──────────┬──────────┘
           │ HTTP (API_URL)
           ▼
┌─────────────────────┐
│  EC2 (FastAPI)      │
│  ├── Neo4j Graph    │
│  ├── SQL Firewall   │
│  └── Query Router   │
└──────────┬──────────┘
           │
     ┌─────┴─────┐
     ▼           ▼
  Athena    S3 Vectors
```

## Prerequisites

- Rosetta SDL FastAPI + Neo4j running on EC2 (port 8000 accessible)
- Python 3.11+, AWS CLI configured
- `bedrock-agentcore-starter-toolkit` installed

## Time: ~15 minutes

---
## Step 1: Install Dependencies

In [ ]:
!pip install -r requirements.txt --quiet
!pip install bedrock-agentcore-starter-toolkit --quiet

---
## Step 2: Configure Environment

Auto-discovers the ALB DNS from the CDK CloudFormation stack outputs — no manual IP needed.

In [ ]:
import boto3
import json
import sys
import os
import logging
import uuid
from pathlib import Path
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
import ac_utils as utils

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

boto_session = boto3.session.Session()
sts_client = boto3.client("sts")
identity = sts_client.get_caller_identity()
ACCOUNT_ID = identity["Account"]
REGION = boto_session.region_name

# === Auto-discover ALB DNS from CDK stack outputs ===
STACK_NAME = "RosettaSdlStack"
cfn = boto3.client("cloudformation", region_name=REGION)

try:
    outputs = cfn.describe_stacks(StackName=STACK_NAME)["Stacks"][0]["Outputs"]
    alb_dns = next(o["OutputValue"] for o in outputs if o["OutputKey"] == "AlbDnsName")
    API_URL = f"http://{alb_dns}"
    print(f"Auto-discovered ALB from {STACK_NAME} stack")
except Exception as e:
    print(f"WARNING: Could not auto-discover ALB from CloudFormation: {e}")
    print("Falling back to manual configuration — set API_URL below:")
    API_URL = "http://REPLACE_WITH_ALB_DNS:8000"

print(f"\nRegion: {REGION}")
print(f"Account: {ACCOUNT_ID}")
print(f"API URL: {API_URL}")

---
## Step 3: Verify Rosetta SDL is reachable

Make sure the FastAPI service on EC2 is running and accessible.

In [ ]:
import httpx

try:
    resp = httpx.get(f"{API_URL}/health", timeout=10)
    health = resp.json()
    print(f"FastAPI status: {health.get('status')}")
    print(f"Neo4j: {health.get('neo4j')}")
    if health.get('neo4j') != 'connected':
        print("WARNING: Neo4j is not connected. Seed the graph first.")
    else:
        print("Rosetta SDL is healthy and ready.")
except Exception as e:
    print(f"ERROR: Cannot reach {API_URL}/health")
    print(f"  {e}")
    print(f"  Make sure EC2 security group allows inbound on port 8000")

---
## Step 4: Create IAM Roles

Two roles needed:
1. **Gateway Role** — allows Gateway to route requests to Runtime
2. **Runtime Role** — allows MCP server container to run (no data permissions needed, FastAPI handles that)

In [ ]:
# Gateway IAM Role
print("Creating Gateway IAM role...")
gateway_role_arn = utils.create_agentcore_gateway_role("rosetta-sdl-gw")
print(f"Gateway Role ARN: {gateway_role_arn}")

In [ ]:
# Runtime IAM Role
print("Creating Runtime IAM role...")
runtime_role_arn = utils.create_agentcore_runtime_role("rosetta-sdl")
print(f"Runtime Role ARN: {runtime_role_arn}")

---
## Step 5: Create Cognito Pools

Two-tier authentication:
1. **Gateway Pool** (inbound) — clients get JWT tokens to call the Gateway
2. **Runtime Pool** (outbound) — Gateway gets OAuth2 tokens to call Runtime

In [ ]:
cognito = boto3.client("cognito-idp", region_name=REGION)

# --- Gateway Cognito Pool (Inbound Auth) ---
print("Setting up Gateway Cognito Pool...")

GW_POOL_NAME = "rosetta-sdl-gateway-pool"
GW_RS_ID = "rosetta-sdl-gateway"
GW_RS_NAME = "Rosetta SDL Gateway"
GW_CLIENT_NAME = "rosetta-sdl-gateway-client"
GW_SCOPES = [{"ScopeName": "invoke", "ScopeDescription": "Invoke the Rosetta SDL gateway"}]

gw_pool_id = utils.get_or_create_user_pool(cognito, GW_POOL_NAME)
utils.get_or_create_resource_server(cognito, gw_pool_id, GW_RS_ID, GW_RS_NAME, GW_SCOPES)
gw_scope_names = [f"{GW_RS_ID}/{s['ScopeName']}" for s in GW_SCOPES]
gw_client_id, gw_client_secret = utils.get_or_create_m2m_client(cognito, gw_pool_id, GW_CLIENT_NAME, GW_RS_ID, gw_scope_names)
gw_discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{gw_pool_id}/.well-known/openid-configuration"
gw_scope_string = " ".join(gw_scope_names)

gw_cognito_config = {
    "user_pool_id": gw_pool_id,
    "client_id": gw_client_id,
    "client_secret": gw_client_secret,
    "discovery_url": gw_discovery_url,
    "scope_string": gw_scope_string,
}

print(f"Gateway Pool ID: {gw_pool_id}")
print(f"Gateway Client ID: {gw_client_id}")

In [ ]:
# --- Runtime Cognito Pool (Outbound Auth) ---
print("Setting up Runtime Cognito Pool...")

RT_POOL_NAME = "rosetta-sdl-runtime-pool"
RT_RS_ID = "rosetta-sdl-runtime"
RT_RS_NAME = "Rosetta SDL Runtime"
RT_CLIENT_NAME = "rosetta-sdl-runtime-client"
RT_SCOPES = [{"ScopeName": "invoke", "ScopeDescription": "Invoke the Rosetta SDL runtime"}]

rt_pool_id = utils.get_or_create_user_pool(cognito, RT_POOL_NAME)
utils.get_or_create_resource_server(cognito, rt_pool_id, RT_RS_ID, RT_RS_NAME, RT_SCOPES)
rt_scope_names = [f"{RT_RS_ID}/{s['ScopeName']}" for s in RT_SCOPES]
rt_client_id, rt_client_secret = utils.get_or_create_m2m_client(cognito, rt_pool_id, RT_CLIENT_NAME, RT_RS_ID, rt_scope_names)
rt_discovery_url = f"https://cognito-idp.{REGION}.amazonaws.com/{rt_pool_id}/.well-known/openid-configuration"
rt_scope_string = " ".join(rt_scope_names)

runtime_cognito_config = {
    "user_pool_id": rt_pool_id,
    "client_id": rt_client_id,
    "client_secret": rt_client_secret,
    "discovery_url": rt_discovery_url,
    "scope_string": rt_scope_string,
}

print(f"Runtime Pool ID: {rt_pool_id}")
print(f"Runtime Client ID: {rt_client_id}")

---
## Step 6: Create AgentCore Gateway

The Gateway is the MCP entry point. It validates JWT tokens and routes requests
to the Rosetta SDL Runtime target.

In [ ]:
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
GATEWAY_NAME = "rosetta-sdl-gateway"

# Check if gateway already exists
gateway_id = None
gateway_url = None
try:
    for gw in gateway_client.list_gateways().get("items", []):
        if gw.get("name") == GATEWAY_NAME:
            gateway_id = gw["gatewayId"]
            details = gateway_client.get_gateway(gatewayIdentifier=gateway_id)
            gateway_url = details["gatewayUrl"]
            print(f"Using existing gateway: {gateway_id}")
            break
except Exception as e:
    print(f"Could not list gateways: {e}")

if not gateway_id:
    print(f"Creating gateway: {GATEWAY_NAME}")
    create_resp = gateway_client.create_gateway(
        name=GATEWAY_NAME,
        roleArn=gateway_role_arn,
        protocolType="MCP",
        protocolConfiguration={
            "mcp": {
                "supportedVersions": ["2025-03-26"],
                "searchType": "SEMANTIC",
            }
        },
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "allowedClients": [gw_client_id],
                "discoveryUrl": gw_discovery_url,
            }
        },
        description="Rosetta SDL — Semantic Data Layer MCP Gateway",
    )
    gateway_id = create_resp["gatewayId"]
    gateway_url = create_resp["gatewayUrl"]

print(f"Gateway ID: {gateway_id}")
print(f"Gateway URL: {gateway_url}")

---
## Step 7: Deploy Rosetta SDL MCP Server to Runtime

This packages `rosetta_mcp.py` into a Docker container, pushes to ECR,
and deploys to AgentCore Runtime.

The MCP server receives tool calls and proxies them to FastAPI on EC2.

**This takes 5-10 minutes** (container build + deploy).

In [ ]:
print("Deploying Rosetta SDL MCP Server to AgentCore Runtime...")
print(f"  API_URL will be set to: {API_URL}")
print("  This may take 5-10 minutes...")

# Save current directory
original_dir = os.getcwd()
script_dir = Path.cwd()
os.chdir(script_dir)

try:
    agentcore_runtime = Runtime()

    auth_config = {
        "customJWTAuthorizer": {
            "allowedClients": [rt_client_id],
            "discoveryUrl": rt_discovery_url,
        }
    }

    agentcore_runtime.configure(
        entrypoint="rosetta_mcp.py",
        execution_role=runtime_role_arn,
        auto_create_ecr=True,
        requirements_file="requirements.txt",
        non_interactive=True,
        region=REGION,
        authorizer_configuration=auth_config,
        protocol="MCP",
        agent_name="rosetta_sdl_mcp",
    )

    launch_result = agentcore_runtime.launch(
        auto_update_on_conflict=True,
        env_vars={"API_URL": API_URL},
    )

    agent_arn = launch_result.agent_arn
    agent_id = launch_result.agent_id
    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    agent_url = f"https://bedrock-agentcore.{REGION}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

    print(f"\nRosetta SDL MCP deployed!")
    print(f"  Agent ARN: {agent_arn}")
    print(f"  Agent ID: {agent_id}")
    print(f"  Agent URL: {agent_url}")
finally:
    os.chdir(original_dir)

---
## Step 8: Create OAuth2 Credential Provider

Enables the Gateway to authenticate with the Runtime using OAuth2 client credentials.

In [ ]:
print("Creating OAuth2 credential provider...")

identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

cognito_provider = identity_client.create_oauth2_credential_provider(
    name="rosetta-sdl-identity",
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "oauthDiscovery": {"discoveryUrl": rt_discovery_url},
            "clientId": rt_client_id,
            "clientSecret": rt_client_secret,
        }
    },
)

credential_provider_arn = cognito_provider["credentialProviderArn"]
print(f"Credential Provider ARN: {credential_provider_arn}")

---
## Step 9: Create Gateway Target

Connect the Gateway to the Rosetta SDL MCP Runtime.

In [ ]:
TARGET_NAME = "rosetta-sdl-mcp-target"

# Check if target exists
target_id = None
try:
    for target in gateway_client.list_gateway_targets(gatewayIdentifier=gateway_id).get("items", []):
        if target.get("name") == TARGET_NAME:
            target_id = target["targetId"]
            print(f"Using existing target: {target_id}")
            break
except Exception as e:
    print(f"Could not list targets: {e}")

if not target_id:
    print(f"Creating gateway target: {TARGET_NAME}")
    target_resp = gateway_client.create_gateway_target(
        name=TARGET_NAME,
        gatewayIdentifier=gateway_id,
        targetConfiguration={
            "mcp": {"mcpServer": {"endpoint": agent_url}}
        },
        credentialProviderConfigurations=[
            {
                "credentialProviderType": "OAUTH",
                "credentialProvider": {
                    "oauthCredentialProvider": {
                        "providerArn": credential_provider_arn,
                        "scopes": [rt_scope_string],
                    }
                },
            }
        ],
    )
    target_id = target_resp["targetId"]

print(f"Target ID: {target_id}")

---
## Step 10: Verify Deployment

In [ ]:
print("Verifying gateway targets...")
targets = gateway_client.list_gateway_targets(gatewayIdentifier=gateway_id).get("items", [])
for t in targets:
    print(f"  {t.get('name')}: {t.get('status')}")

print(f"\nTotal targets: {len(targets)}")

---
## Step 11: Save Deployment Info

In [ ]:
# Token URL for clients
gw_pool_domain = gw_pool_id.lower().replace("_", "")
token_url = f"https://{gw_pool_domain}.auth.{REGION}.amazoncognito.com/oauth2/token"

print("=" * 70)
print("ROSETTA SDL - AGENTCORE DEPLOYMENT INFO")
print("=" * 70)
print(f"\nGateway URL:   {gateway_url}")
print(f"Gateway ID:    {gateway_id}")
print(f"\nToken URL:     {token_url}")
print(f"Client ID:     {gw_client_id}")
print(f"Client Secret: {gw_client_secret}")
print(f"Scope:         {gw_scope_string}")
print(f"\nAgent ARN:     {agent_arn}")
print(f"Agent ID:      {agent_id}")
print(f"Target ID:     {target_id}")
print(f"\nFastAPI URL:   {API_URL}")

deployment_info = {
    "gateway": {"gateway_id": gateway_id, "gateway_url": gateway_url},
    "gateway_role_arn": gateway_role_arn,
    "runtime_role_arn": runtime_role_arn,
    "auth": {
        "token_url": token_url,
        "client_id": gw_client_id,
        "client_secret": gw_client_secret,
        "scope": gw_scope_string,
    },
    "agent": {"arn": agent_arn, "id": agent_id, "url": agent_url},
    "target_id": target_id,
    "credential_provider_arn": credential_provider_arn,
    "fastapi_url": API_URL,
}

with open("deployment_info.json", "w") as f:
    json.dump(deployment_info, f, indent=2)

print(f"\nSaved to deployment_info.json")
print("=" * 70)

---
## Step 12: Test the Gateway

Get a token and call the Gateway to list available tools.

In [ ]:
import requests

# Get access token
token_resp = requests.post(
    token_url,
    data={
        "grant_type": "client_credentials",
        "client_id": gw_client_id,
        "client_secret": gw_client_secret,
        "scope": gw_scope_string,
    },
    headers={"Content-Type": "application/x-www-form-urlencoded"},
)

if token_resp.status_code == 200:
    access_token = token_resp.json()["access_token"]
    print(f"Got access token (expires in {token_resp.json().get('expires_in')}s)")
    print(f"Token prefix: {access_token[:20]}...")
else:
    print(f"Token request failed: {token_resp.status_code}")
    print(token_resp.text)

In [ ]:
# Call the gateway to list tools
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}

# MCP tools/list request
mcp_request = {
    "jsonrpc": "2.0",
    "id": 1,
    "method": "tools/list",
}

resp = requests.post(gateway_url, json=mcp_request, headers=headers)
if resp.status_code == 200:
    result = resp.json()
    tools = result.get("result", {}).get("tools", [])
    print(f"Available tools ({len(tools)}):")
    for tool in tools:
        print(f"  - {tool['name']}: {tool.get('description', '')[:80]}")
else:
    print(f"Gateway call failed: {resp.status_code}")
    print(resp.text)

---
## Cleanup (Optional)

Run this to tear down all AgentCore resources.

In [ ]:
# UNCOMMENT TO CLEANUP
# print("Cleaning up...")
#
# # Delete gateway targets
# try:
#     for target in gateway_client.list_gateway_targets(gatewayIdentifier=gateway_id).get("items", []):
#         gateway_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target["targetId"])
#         print(f"  Deleted target: {target['name']}")
# except Exception as e:
#     print(f"  Target cleanup error: {e}")
#
# # Delete gateway
# try:
#     gateway_client.delete_gateway(gatewayIdentifier=gateway_id)
#     print(f"  Deleted gateway: {gateway_id}")
# except Exception as e:
#     print(f"  Gateway cleanup error: {e}")
#
# # Delete Cognito pools
# try:
#     cognito.delete_user_pool(UserPoolId=gw_pool_id)
#     print(f"  Deleted gateway pool: {gw_pool_id}")
#     cognito.delete_user_pool(UserPoolId=rt_pool_id)
#     print(f"  Deleted runtime pool: {rt_pool_id}")
# except Exception as e:
#     print(f"  Cognito cleanup error: {e}")
#
# print("Cleanup complete.")